# 11 - SHACL Validator (F6.2)

Validates RDF instance data against SHACL shapes parsed in notebook 10.

**Input**: 
- `bronze_triples` - Instance data to validate
- `silver_shacl_shapes` - Parsed constraints from SHACL shapes

**Output**: 
- `silver_validation_results` Delta table
- `/config/validation_report.json` summary

## Constraints Validated

| Constraint | Description | Implementation |
|------------|-------------|----------------|
| sh:minCount | Minimum property occurrences | ✅ Count check |
| sh:maxCount | Maximum property occurrences | ✅ Count check |
| sh:datatype | Required XSD datatype | ✅ Type match |
| sh:class | Object must be instance of class | ✅ Type lookup |
| sh:pattern | Regex pattern for values | ✅ Regex match |
| sh:minLength / sh:maxLength | String length bounds | ✅ Length check |
| sh:nodeKind | IRI, Literal, BlankNode | ✅ Node type check |

## Severity Levels

| Level | Meaning | Default Action |
|-------|---------|----------------|
| sh:Violation | Must fix before loading | Block |
| sh:Warning | Should fix, data may be incomplete | Continue with warning |
| sh:Info | Informational, no action needed | Log only |

In [ ]:
# Configuration
TRIPLES_TABLE = "dbo.bronze_triples"
SHAPES_TABLE = "dbo.silver_shacl_shapes"
OUTPUT_TABLE = "dbo.silver_validation_results"
OUTPUT_JSON = "/lakehouse/default/Files/config/validation_report.json"

# Validation settings
SEVERITY_THRESHOLD = "Warning"  # "Violation", "Warning", or "Info"
FAIL_ON_VIOLATION = True        # Stop if any violations found
MAX_VIOLATIONS_DISPLAY = 100    # Limit output for large datasets

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from datetime import datetime
import json
import re

# Standard namespaces
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"
XSD = "http://www.w3.org/2001/XMLSchema#"
SH = "http://www.w3.org/ns/shacl#"

# SHACL severity levels (ordered by severity)
SEVERITY_ORDER = {
    SH + "Violation": 1,
    SH + "Warning": 2,
    SH + "Info": 3,
    "Violation": 1,
    "Warning": 2,
    "Info": 3,
}

def severity_passes_threshold(severity: str, threshold: str) -> bool:
    """Check if severity meets or exceeds threshold."""
    sev_level = SEVERITY_ORDER.get(severity, 1)  # Default to Violation
    thresh_level = SEVERITY_ORDER.get(threshold, 2)  # Default to Warning
    return sev_level <= thresh_level

In [ ]:
@dataclass
class Violation:
    """A SHACL constraint violation."""
    focus_node: str
    shape_name: str
    property_path: str
    constraint_type: str
    expected: str
    actual: str
    message: str
    severity: str = "Violation"
    
    def to_dict(self) -> dict:
        return {
            "focusNode": self.focus_node,
            "shapeName": self.shape_name,
            "propertyPath": self.property_path,
            "constraintType": self.constraint_type,
            "expected": self.expected,
            "actual": self.actual,
            "message": self.message,
            "severity": self.severity
        }


@dataclass 
class ValidationResult:
    """Overall validation result."""
    conforms: bool
    violations: List[Violation] = field(default_factory=list)
    warnings: List[Violation] = field(default_factory=list)
    info: List[Violation] = field(default_factory=list)
    instances_checked: int = 0
    constraints_checked: int = 0
    
    def add_violation(self, v: Violation):
        if v.severity in ["Violation", SH + "Violation"]:
            self.violations.append(v)
            self.conforms = False
        elif v.severity in ["Warning", SH + "Warning"]:
            self.warnings.append(v)
        else:
            self.info.append(v)
    
    def to_dict(self) -> dict:
        return {
            "conforms": self.conforms,
            "violationCount": len(self.violations),
            "warningCount": len(self.warnings),
            "infoCount": len(self.info),
            "instancesChecked": self.instances_checked,
            "constraintsChecked": self.constraints_checked,
            "violations": [v.to_dict() for v in self.violations[:MAX_VIOLATIONS_DISPLAY]],
            "warnings": [v.to_dict() for v in self.warnings[:MAX_VIOLATIONS_DISPLAY]],
        }

In [ ]:
# Load data
df_triples = spark.table(TRIPLES_TABLE)
df_shapes = spark.table(SHAPES_TABLE)

triple_count = df_triples.count()
shape_count = df_shapes.count()

print(f"Loaded {triple_count:,} triples from '{TRIPLES_TABLE}'")
print(f"Loaded {shape_count:,} shape constraints from '{SHAPES_TABLE}'")

# Show shapes summary
print("\nShapes with constraints:")
df_shapes.filter(F.col("target_class").isNotNull()) \
    .groupBy("shape_name", "target_class") \
    .count() \
    .orderBy("shape_name") \
    .show(30, truncate=False)

In [ ]:
def extract_local_name(uri: str) -> str:
    """Extract local name from URI."""
    if uri is None:
        return None
    if "#" in uri:
        return uri.split("#")[-1]
    if "/" in uri:
        return uri.split("/")[-1]
    return uri


def get_instances_of_class(df_triples, class_uri: str) -> List[str]:
    """Get all instances (subjects) that are rdf:type of given class."""
    instances = df_triples.filter(
        (F.col("predicate") == RDF_TYPE) &
        (F.col("object") == class_uri)
    ).select("subject").distinct().collect()
    return [row.subject for row in instances]


def get_property_values(df_triples, subject: str, predicate: str) -> List[Tuple[str, str, str]]:
    """Get all (object, object_type, datatype) tuples for subject-predicate pair."""
    rows = df_triples.filter(
        (F.col("subject") == subject) &
        (F.col("predicate") == predicate)
    ).select("object", "object_type", "datatype").collect()
    return [(row.object, row.object_type, row.datatype) for row in rows]


def get_instance_types(df_triples, instance_uri: str) -> List[str]:
    """Get all rdf:type values for an instance."""
    rows = df_triples.filter(
        (F.col("subject") == instance_uri) &
        (F.col("predicate") == RDF_TYPE)
    ).select("object").collect()
    return [row.object for row in rows]

In [ ]:
def validate_min_count(values: List, min_count: int, prop_name: str, instance: str, shape_name: str, severity: str) -> Optional[Violation]:
    """Validate sh:minCount constraint."""
    actual_count = len(values)
    if actual_count < min_count:
        return Violation(
            focus_node=instance,
            shape_name=shape_name,
            property_path=prop_name,
            constraint_type="minCount",
            expected=f">= {min_count}",
            actual=str(actual_count),
            message=f"Property '{extract_local_name(prop_name)}' has {actual_count} values, minimum required is {min_count}",
            severity=severity or "Violation"
        )
    return None


def validate_max_count(values: List, max_count: int, prop_name: str, instance: str, shape_name: str, severity: str) -> Optional[Violation]:
    """Validate sh:maxCount constraint."""
    actual_count = len(values)
    if actual_count > max_count:
        return Violation(
            focus_node=instance,
            shape_name=shape_name,
            property_path=prop_name,
            constraint_type="maxCount",
            expected=f"<= {max_count}",
            actual=str(actual_count),
            message=f"Property '{extract_local_name(prop_name)}' has {actual_count} values, maximum allowed is {max_count}",
            severity=severity or "Violation"
        )
    return None


def validate_datatype(values: List[Tuple], expected_datatype: str, prop_name: str, instance: str, shape_name: str, severity: str) -> List[Violation]:
    """Validate sh:datatype constraint for all values."""
    violations = []
    for obj, obj_type, actual_datatype in values:
        if obj_type != "literal":
            violations.append(Violation(
                focus_node=instance,
                shape_name=shape_name,
                property_path=prop_name,
                constraint_type="datatype",
                expected=f"literal with datatype {extract_local_name(expected_datatype)}",
                actual=f"{obj_type}: {obj[:50]}",
                message=f"Expected literal, got {obj_type}",
                severity=severity or "Violation"
            ))
        elif actual_datatype and actual_datatype != expected_datatype:
            # Only flag if datatype is specified and doesn't match
            violations.append(Violation(
                focus_node=instance,
                shape_name=shape_name,
                property_path=prop_name,
                constraint_type="datatype",
                expected=extract_local_name(expected_datatype),
                actual=extract_local_name(actual_datatype) if actual_datatype else "untyped",
                message=f"Expected datatype {extract_local_name(expected_datatype)}, got {extract_local_name(actual_datatype)}",
                severity=severity or "Warning"  # Datatype mismatch often a warning
            ))
    return violations


def validate_class_constraint(df_triples, values: List[Tuple], expected_class: str, prop_name: str, instance: str, shape_name: str, severity: str) -> List[Violation]:
    """Validate sh:class constraint - object must be instance of expected class."""
    violations = []
    for obj, obj_type, _ in values:
        if obj_type == "literal":
            violations.append(Violation(
                focus_node=instance,
                shape_name=shape_name,
                property_path=prop_name,
                constraint_type="class",
                expected=f"instance of {extract_local_name(expected_class)}",
                actual=f"literal: {obj[:50]}",
                message=f"Expected IRI instance of {extract_local_name(expected_class)}, got literal",
                severity=severity or "Violation"
            ))
        else:
            # Check if object is instance of expected class
            obj_types = get_instance_types(df_triples, obj)
            if expected_class not in obj_types:
                # Also check if object types include subclasses (simplified - just check direct type)
                type_names = [extract_local_name(t) for t in obj_types] if obj_types else ["(no type)"]
                violations.append(Violation(
                    focus_node=instance,
                    shape_name=shape_name,
                    property_path=prop_name,
                    constraint_type="class",
                    expected=extract_local_name(expected_class),
                    actual=", ".join(type_names),
                    message=f"Object '{extract_local_name(obj)}' is not instance of {extract_local_name(expected_class)}",
                    severity=severity or "Warning"  # Often schema doesn't include all instances
                ))
    return violations


def validate_pattern(values: List[Tuple], pattern: str, prop_name: str, instance: str, shape_name: str, severity: str) -> List[Violation]:
    """Validate sh:pattern constraint - value must match regex."""
    violations = []
    try:
        regex = re.compile(pattern)
        for obj, obj_type, _ in values:
            if obj_type == "literal" and not regex.search(obj):
                violations.append(Violation(
                    focus_node=instance,
                    shape_name=shape_name,
                    property_path=prop_name,
                    constraint_type="pattern",
                    expected=f"matches /{pattern}/",
                    actual=obj[:50],
                    message=f"Value '{obj[:30]}' does not match pattern '{pattern}'",
                    severity=severity or "Violation"
                ))
    except re.error:
        pass  # Invalid regex pattern in shape
    return violations


def validate_node_kind(values: List[Tuple], expected_kind: str, prop_name: str, instance: str, shape_name: str, severity: str) -> List[Violation]:
    """Validate sh:nodeKind constraint."""
    violations = []
    kind_name = extract_local_name(expected_kind)
    
    for obj, obj_type, _ in values:
        actual_kind = None
        if obj_type == "uri":
            actual_kind = "IRI"
        elif obj_type == "literal":
            actual_kind = "Literal"
        elif obj_type == "bnode":
            actual_kind = "BlankNode"
        
        # Map SHACL nodeKind values
        expected_kinds = {
            "IRI": ["IRI", "BlankNodeOrIRI", "IRIOrLiteral"],
            "Literal": ["Literal", "IRIOrLiteral", "BlankNodeOrLiteral"],
            "BlankNode": ["BlankNode", "BlankNodeOrIRI", "BlankNodeOrLiteral"],
        }
        
        allowed = expected_kinds.get(actual_kind, [])
        if kind_name not in allowed and actual_kind != kind_name:
            violations.append(Violation(
                focus_node=instance,
                shape_name=shape_name,
                property_path=prop_name,
                constraint_type="nodeKind",
                expected=kind_name,
                actual=actual_kind,
                message=f"Expected {kind_name}, got {actual_kind}",
                severity=severity or "Violation"
            ))
    return violations

In [ ]:
def validate_constraint(df_triples, instance: str, constraint_row, result: ValidationResult):
    """Validate a single constraint against an instance."""
    prop_path = constraint_row.property_path
    shape_name = constraint_row.shape_name
    
    # Get actual values for this property
    values = get_property_values(df_triples, instance, prop_path)
    result.constraints_checked += 1
    
    # Default severity
    severity = "Violation"
    
    # Cardinality constraints
    if constraint_row.min_count is not None:
        v = validate_min_count(values, constraint_row.min_count, prop_path, instance, shape_name, severity)
        if v:
            result.add_violation(v)
    
    if constraint_row.max_count is not None:
        v = validate_max_count(values, constraint_row.max_count, prop_path, instance, shape_name, severity)
        if v:
            result.add_violation(v)
    
    # Only check other constraints if there are values
    if not values:
        return
    
    # Datatype constraint
    if constraint_row.datatype:
        for v in validate_datatype(values, constraint_row.datatype, prop_path, instance, shape_name, severity):
            result.add_violation(v)
    
    # Class constraint
    if constraint_row.class_constraint:
        for v in validate_class_constraint(df_triples, values, constraint_row.class_constraint, prop_path, instance, shape_name, "Warning"):
            result.add_violation(v)
    
    # Pattern constraint
    if constraint_row.pattern:
        for v in validate_pattern(values, constraint_row.pattern, prop_path, instance, shape_name, severity):
            result.add_violation(v)
    
    # NodeKind constraint
    if constraint_row.node_kind:
        for v in validate_node_kind(values, constraint_row.node_kind, prop_path, instance, shape_name, severity):
            result.add_violation(v)

In [ ]:
# Main validation loop
print("=" * 60)
print("SHACL VALIDATION")
print("=" * 60)

result = ValidationResult(conforms=True)

# Get shapes grouped by target class
shapes_by_class = df_shapes.filter(
    F.col("target_class").isNotNull()
).collect()

# Group constraints by target class
class_constraints = {}
for row in shapes_by_class:
    target = row.target_class
    if target not in class_constraints:
        class_constraints[target] = []
    class_constraints[target].append(row)

print(f"\nValidating {len(class_constraints)} target classes...")

# Validate each class
for target_class, constraints in class_constraints.items():
    class_name = extract_local_name(target_class)
    instances = get_instances_of_class(df_triples, target_class)
    
    if not instances:
        print(f"  {class_name}: 0 instances (skipped)")
        continue
    
    print(f"  {class_name}: {len(instances)} instances, {len(constraints)} constraints")
    result.instances_checked += len(instances)
    
    # Validate each instance against all constraints for this class
    for instance in instances:
        for constraint in constraints:
            validate_constraint(df_triples, instance, constraint, result)

print(f"\nValidation complete.")
print(f"  Instances checked: {result.instances_checked}")
print(f"  Constraints checked: {result.constraints_checked}")

In [ ]:
# Display results
print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

if result.conforms:
    print("\n✅ DATA CONFORMS - No violations found")
else:
    print(f"\n❌ VALIDATION FAILED - {len(result.violations)} violation(s)")

print(f"\nSummary:")
print(f"  Violations: {len(result.violations)}")
print(f"  Warnings: {len(result.warnings)}")
print(f"  Info: {len(result.info)}")

# Show violations
if result.violations:
    print("\n" + "-" * 60)
    print("VIOLATIONS (must fix)")
    print("-" * 60)
    for i, v in enumerate(result.violations[:20]):
        print(f"\n[{i+1}] {v.shape_name} / {extract_local_name(v.property_path)}")
        print(f"    Focus: {extract_local_name(v.focus_node)}")
        print(f"    Constraint: {v.constraint_type}")
        print(f"    Expected: {v.expected}")
        print(f"    Actual: {v.actual}")
        print(f"    Message: {v.message}")
    
    if len(result.violations) > 20:
        print(f"\n... and {len(result.violations) - 20} more violations")

# Show warnings
if result.warnings:
    print("\n" + "-" * 60)
    print("WARNINGS (should review)")
    print("-" * 60)
    for i, v in enumerate(result.warnings[:10]):
        print(f"\n[{i+1}] {v.shape_name} / {extract_local_name(v.property_path)}")
        print(f"    Focus: {extract_local_name(v.focus_node)}")
        print(f"    Message: {v.message}")
    
    if len(result.warnings) > 10:
        print(f"\n... and {len(result.warnings) - 10} more warnings")

In [ ]:
# Save results to Delta table
violation_schema = StructType([
    StructField("focus_node", StringType(), True),
    StructField("shape_name", StringType(), True),
    StructField("property_path", StringType(), True),
    StructField("constraint_type", StringType(), True),
    StructField("expected", StringType(), True),
    StructField("actual", StringType(), True),
    StructField("message", StringType(), True),
    StructField("severity", StringType(), True),
])

all_issues = result.violations + result.warnings + result.info

if all_issues:
    from pyspark.sql import Row
    rows = [Row(
        focus_node=v.focus_node,
        shape_name=v.shape_name,
        property_path=v.property_path,
        constraint_type=v.constraint_type,
        expected=v.expected,
        actual=v.actual,
        message=v.message,
        severity=v.severity
    ) for v in all_issues]
    
    df_results = spark.createDataFrame(rows, schema=violation_schema)
    df_results.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)
    print(f"\nSaved {len(all_issues)} validation issues to '{OUTPUT_TABLE}'")
else:
    # Create empty table
    df_results = spark.createDataFrame([], schema=violation_schema)
    df_results.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)
    print(f"\nNo issues found - created empty '{OUTPUT_TABLE}' table")

In [ ]:
# Save JSON report
report = {
    "validatedAt": datetime.now().isoformat(),
    "sourceTables": {
        "triples": TRIPLES_TABLE,
        "shapes": SHAPES_TABLE
    },
    "settings": {
        "severityThreshold": SEVERITY_THRESHOLD,
        "failOnViolation": FAIL_ON_VIOLATION
    },
    "result": result.to_dict()
}

with open(OUTPUT_JSON, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\nValidation report saved to: {OUTPUT_JSON}")

In [ ]:
# Final status
print("\n" + "=" * 60)
print("F6.2 SHACL VALIDATOR COMPLETE")
print("=" * 60)

print(f"\nOutputs:")
print(f"  - Delta table: {OUTPUT_TABLE}")
print(f"  - JSON report: {OUTPUT_JSON}")

print(f"\nResult: {'✅ CONFORMS' if result.conforms else '❌ VIOLATIONS FOUND'}")
print(f"  - Violations: {len(result.violations)}")
print(f"  - Warnings: {len(result.warnings)}")

if FAIL_ON_VIOLATION and not result.conforms:
    print("\n⚠️  FAIL_ON_VIOLATION is True - pipeline should stop here")
    print("   Set FAIL_ON_VIOLATION = False to continue with warnings")